In [3]:
%%capture
% pip install openpyxl
import pandas as pd
import numpy as np
from pathlib import Path
import openpyxl
import statsmodels.api as sm

# Optional für Plots
import matplotlib.pyplot as plt


In [5]:
# save to csv
df_races = pd.read_csv("csv/races.csv")
df_laps = pd.read_csv("csv/laps.csv") 

In [8]:
hilfe_map = {
    1: "No Help",
    2: "With Driving Coach",
    3: "With Ideal Driving Line"
}

df_races["help_label"] = df_races["help"].map(hilfe_map)
df_laps["help_label"] = df_laps["help"].map(hilfe_map)


In [9]:
grouped = (
    df_races
    .groupby(["help", "racetrack"])["total_duration"]
    .mean()
    .reset_index()
)

grouped

,help,racetrack,total_duration
0,1.0,1.0,296.488750
1,1.0,2.0,379.925000
2,1.0,3.0,349.718571
3,2.0,1.0,263.850000
4,2.0,2.0,420.692500
5,2.0,3.0,391.995000
6,3.0,1.0,284.466667
7,3.0,2.0,367.702857
8,3.0,3.0,406.722500


In [10]:
mean_total = (
    df_races
    .groupby("help_label")["total_duration"]
    .mean()
    .reset_index()
)

plt.figure()
plt.bar(mean_total["help_label"], mean_total["total_duration"])
plt.xlabel("Help condition")
plt.ylabel("Average total duration (s)")
plt.title("Total duration by help condition")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("pictures/total_duration_by_help_condition.png")
plt.close()

In [11]:
mean_lap = (
    df_laps
    .groupby(["help_label", "lap"])["duration"]
    .mean()
    .reset_index()
)

plt.figure()

for hilfe in mean_lap["help_label"].unique():
    subset = mean_lap[mean_lap["help_label"] == hilfe]
    plt.plot(subset["lap"], subset["duration"], marker="o", label=hilfe)

plt.xlabel("Lap number")
plt.ylabel("Average lap duration (s)")
plt.title("Lap-to-Lap Duration by Help Condition")
plt.legend()
plt.tight_layout()
plt.savefig("pictures/lap_improvement_by_help_condition.png")
plt.close()

In [12]:
# Perform one hot encoding and save to csv
df_races_encoded = pd.get_dummies(
    df_races,
    columns=["racetrack", "help", "experience", "race_number"],
    drop_first=True
)

df_laps_encoded = pd.get_dummies(
    df_laps,
    columns=["racetrack", "lap", "help", "experience", "race_number"],
    drop_first=True
)

df_races_encoded.columns = [
    "name",
    "id",
    "total_duration",
    "three_race_duration",
    "help_label",
    "racetrack_2",
    "racetrack_3",
    "help_2",
    "help_3",
    "experience_2",
    "experience_3",
    "race_number_2",
    "race_number_3"
]


df_laps_encoded.columns = [
    "name",
    "id",
    "duration",
    "three_race_duration",
    "help_label",
    "racetrack_2",
    "racetrack_3",
    "lap_2",
    "lap_3",
    "lap_4",
    "lap_5",
    "help_2",
    "help_3",
    "experience_2",
    "experience_3",
    "race_number_2",
    "race_number_3"
]

df_races_encoded.to_csv("csv/races_oneHot.csv", index=False)
df_laps_encoded.to_csv("csv/laps_oneHot.csv", index=False) 

In [13]:
# Fit regression model
X = df_races_encoded[
        [
            "racetrack_2",
            "racetrack_3",
            "help_2",
            "help_3",
            "experience_2",
            "experience_3",
            "race_number_2",
            "race_number_3",
        ]
    ]
y = df_races_encoded["total_duration"]

X = X.astype(float)
y = y.astype(float)

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:         total_duration   R-squared:                       0.737
Model:                            OLS   Adj. R-squared:                  0.698
Method:                 Least Squares   F-statistic:                     18.94
Date:                Sat, 28 Feb 2026   Prob (F-statistic):           3.59e-13
Time:                        16:31:04   Log-Likelihood:                -327.28
No. Observations:                  63   AIC:                             672.6
Df Residuals:                      54   BIC:                             691.9
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const           349.1991     17.569     19.876

In [14]:
# Fit regression model
X = df_races_encoded[
        [
            "help_2",
            "help_3",
        ]
    ]
y = df_races_encoded["total_duration"]

X = X.astype(float)
y = y.astype(float)

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:         total_duration   R-squared:                       0.014
Model:                            OLS   Adj. R-squared:                 -0.019
Method:                 Least Squares   F-statistic:                    0.4298
Date:                Sat, 28 Feb 2026   Prob (F-statistic):              0.653
Time:                        16:31:04   Log-Likelihood:                -368.94
No. Observations:                  63   AIC:                             743.9
Df Residuals:                      60   BIC:                             750.3
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const        338.0710     18.903     17.884      0.0

In [15]:
# Fit regression model
X = df_races_encoded[
        [
            "racetrack_2",
            "racetrack_3",
            "help_2",
            "help_3",
            # "experience_2",
            # "experience_3",
            "race_number_2",
            "race_number_3",
            "three_race_duration"
        ]
    ]
y = df_races_encoded["total_duration"]

X = X.astype(float)
y = y.astype(float)

X = sm.add_constant(X)

model = sm.OLS(y, X).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:         total_duration   R-squared:                       0.938
Model:                            OLS   Adj. R-squared:                  0.930
Method:                 Least Squares   F-statistic:                     118.2
Date:                Sat, 28 Feb 2026   Prob (F-statistic):           8.61e-31
Time:                        16:31:04   Log-Likelihood:                -281.97
No. Observations:                  63   AIC:                             579.9
Df Residuals:                      55   BIC:                             597.1
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                 -66.9043    

In [16]:
# Fit regression model
X = df_laps_encoded[
        [
            "help_2",
            "help_3",
            "lap_2",
            "lap_3",
            "lap_4",
            "lap_5",
        ]
    ]
y = df_laps_encoded["duration"]

X = X.astype(float)
y = y.astype(float)

X = sm.add_constant(X)

# model = sm.OLS(y, X).fit()

formula = "duration ~ help_2 + help_3 + lap_2 + lap_3 + lap_4 + lap_5 + " \
    "lap_2*help_2 + lap_3*help_2 + lap_4*help_2 + lap_5*help_2 + " \
    "lap_2*help_3 + lap_3*help_3 + lap_4*help_3 + lap_5*help_3"
            
# das entspricht help + lap + help:lap für beide Hilfe‑Variablen
model = sm.OLS.from_formula(formula, data=df_laps_encoded).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:               duration   R-squared:                       0.040
Model:                            OLS   Adj. R-squared:                 -0.005
Method:                 Least Squares   F-statistic:                    0.8890
Date:                Sat, 28 Feb 2026   Prob (F-statistic):              0.571
Time:                        16:31:04   Log-Likelihood:                -1358.8
No. Observations:                 315   AIC:                             2748.
Df Residuals:                     300   BIC:                             2804.
Df Model:                          14                                         
Covariance Type:            nonrobust                                         
                                   coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------
Intercept       

In [17]:
# liste der lap‑dummies, sortiert nach nummer
lap_cols = [c for c in df_laps_encoded.columns if c.startswith("lap_")]
lap_cols = sorted(lap_cols, key=lambda s: int(s.split("_")[1]))

def _decode_lap(row):
    for col in lap_cols:
        if row[col] == 1:
            return int(col.split("_")[1])
    # keine Dummy‑Spalte gesetzt -> Basis‑kategorie = lap 1
    return 1

df_laps_encoded["lap"] = df_laps_encoded.apply(_decode_lap, axis=1)

df_laps_encoded.to_csv("csv/laps_oneHot_decoded.csv", index=False)

In [18]:
# Fit regression model
X = df_laps_encoded[
        [
            "help_2",
            "help_3",
            "lap_2",
            "lap_3",
            "lap_4",
            "lap_5",
            "lap"
        ]
    ]
y = df_laps_encoded["duration"]

X = X.astype(float)
y = y.astype(float)

X = sm.add_constant(X)

# model = sm.OLS(y, X).fit()




formula = "duration ~ help_2 + help_3 + lap + lap*help_2 + lap*help_3"
            
# das entspricht help + lap + help:lap für beide Hilfe‑Variablen
model = sm.OLS.from_formula(formula, data=df_laps_encoded).fit()

print(model.summary())


                            OLS Regression Results                            
Dep. Variable:               duration   R-squared:                       0.034
Model:                            OLS   Adj. R-squared:                  0.019
Method:                 Least Squares   F-statistic:                     2.208
Date:                Sat, 28 Feb 2026   Prob (F-statistic):             0.0534
Time:                        16:31:04   Log-Likelihood:                -1359.7
No. Observations:                 315   AIC:                             2731.
Df Residuals:                     309   BIC:                             2754.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             72.5910      4